In [1]:
# ===================== CELL 0 =====================
%pip install langchain-google-genai --upgrade -q
print("✅ Package updated to latest version")

Note: you may need to restart the kernel to use updated packages.
✅ Package updated to latest version


In [2]:
# ===================== CELL 1 =====================
import os
from dotenv import load_dotenv

print("📍 Loading from: Api_Key.env")

load_dotenv("Api_Key.env", override=True)

api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

if api_key and api_key.startswith("AIza"):
    os.environ["GOOGLE_API_KEY"] = api_key
    print("✅ GOOGLE_API_KEY loaded successfully!")
    print("Preview:", api_key[:10] + "..." + api_key[-6:])
else:
    print("❌ KEY NOT FOUND!")

📍 Loading from: Api_Key.env
✅ GOOGLE_API_KEY loaded successfully!
Preview: AIzaSyC6GB...caXJQ4


In [3]:
# ===================== CELL 2 =====================
import os
from langchain_community.document_loaders import PyPDFLoader

print("📍 Current folder:", os.getcwd())

loader = PyPDFLoader("customer_support_arabic_full.pdf")
# If needed: loader = PyPDFLoader(r"C:\Your\Full\Path\customer_support_arabic_full.pdf")

pages = loader.load()
print(f"✅ Loaded {len(pages)} pages")

d:\PROGRAMS\PYTHON\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📍 Current folder: d:\ITI_DATA\Analytics\smarttech-ragcustomer_support_arabic_full.pdf
✅ Loaded 4 pages


In [4]:
# ===================== CELL 3 =====================
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", "●", "○"]
)

docs = text_splitter.split_documents(pages)
print(f"✅ Created {len(docs)} chunks")

✅ Created 9 chunks


In [5]:
# ===================== CELL 4 =====================
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("Run CELL 1 again!")

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=api_key
)

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./chroma_db_gemini"
)

print("✅ Vector Database ready!")

✅ Vector Database ready!


In [6]:
# ===================== CELL 5 =====================
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",          # ← This is the current correct model (March 2026)
    temperature=0,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

prompt = ChatPromptTemplate.from_template(
    """أنت مساعد خدمة العملاء في شركة سمارت تك للإلكترونيات.
أجب باللغة العربية فقط باستخدام السياق أدناه فقط.
إذا لم تجد الإجابة، قل فقط: "لا أعرف هذه المعلومة من دليل الشركة."

السياق:
{context}

السؤال: {question}

الإجابة:"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Gemini RAG is READY! (using gemini-2.5-flash)")

✅ Gemini RAG is READY! (using gemini-2.5-flash)


In [7]:
# ===================== CELL 6 =====================
print("🎉 RAG بجيميني جاهز! اكتب أسئلتك (اكتب 'exit' للخروج)\n")

while True:
    question = input("❓ سؤالك: ")
    if question.lower() in ["exit", "خروج", "quit"]:
        print("👋 تم إغلاق الدردشة")
        break
    response = rag_chain.invoke(question)
    print("\n✅ الإجابة:", response)
    print("-" * 70)

🎉 RAG بجيميني جاهز! اكتب أسئلتك (اكتب 'exit' للخروج)


✅ الإجابة: لا أعرف هذه المعلومة من دليل الشركة.
----------------------------------------------------------------------

✅ الإجابة: نعم قبل عملية الشحن فقط. بعد الشحن، يجب استلام الطرد ثم إرجاعه.
----------------------------------------------------------------------
👋 تم إغلاق الدردشة
